# Kuiper kernel benchmarks

Each table compares the JIT-dispatched verified Kuiper kernels (`kuipy.run`) and
the unverified reference kernels (`kuipy.unverified`) against stock PyTorch, on
the shapes of a Qwen2.5-0.5B decode step.

Times are us/call, `rel-err` is the relative Frobenius norm against the `ref` column.

In [1]:
import torch
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

import kuipy
from kuipy import unverified
from kuipy.benchmarking import bench_matrix

aten = torch.ops.aten
DEV = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Qwen2.5-0.5B-Instruct, decoding at batch 256.
HID, NH, NKV, HEAD_DIM = 896, 14, 2, 64
INTER, VOCAB, BATCH = 4864, 151936, 256
SCALE = HEAD_DIM ** -0.5
ALPHA, BETA = 0.75, 1.5

_g = torch.Generator(device=DEV).manual_seed(0)

def rand(*shape, dtype=torch.bfloat16):
    return torch.randn(*shape, device=DEV, dtype=dtype, generator=_g) * 0.1

torch.cuda.get_device_name(0)

'NVIDIA RTX A6000'

In [2]:
MM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("gate_proj",   BATCH, HID,   INTER),
    ("up_proj",     BATCH, HID,   INTER),
    ("down_proj",   BATCH, INTER, HID),
    ("lm_head",     BATCH, HID,   VOCAB),
    ("square_4096", 4096,  4096,  4096),
]

def mm_inputs(dtype):
    return lambda M, K, N: ((rand(M, K, dtype=dtype), rand(K, N, dtype=dtype)), {})

# gemm_tc is addmm-shaped; a bias-free matmul is just the absent epilogue term.
def gemm_tc_mm(A, B):
    return unverified.gemm_tc(None, A, B, beta=0.0, alpha=1.0)

MNK = lambda M, K, N: (M, K, N)
GEMM_FLOPS = lambda M, K, N: 2 * M * N * K

GEMM_CASES = [
    ("o_proj",      BATCH, HID,   HID),
    ("down_proj",   BATCH, INTER, HID),
    ("square_4096", 4096,  4096,  4096),
]

def addmm_inputs(dtype):
    return lambda M, K, N: ((rand(M, N, dtype=dtype), rand(M, K, dtype=dtype),
                             rand(K, N, dtype=dtype)), {"beta": BETA, "alpha": ALPHA})

_kuiper_sdpa = kuipy.run(aten._scaled_dot_product_efficient_attention.default)

def kuiper_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    return _kuiper_sdpa(q, k, v, attn_mask, False, 0.0, is_causal, scale=scale)[0]

def cudnn_sdpa(q, k, v, attn_mask=None, is_causal=False, scale=None):
    with sdpa_kernel(SDPBackend.CUDNN_ATTENTION):
        return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                              is_causal=is_causal, scale=scale,
                                              enable_gqa=True)

def attn_flops(sq, sk):
    return 4 * BATCH * NH * sq * sk * HEAD_DIM

DECODE_CASES = [(f"ctx_{c}", 1, c) for c in (128, 512, 1024, 16384)]

def decode_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"attn_mask": torch.zeros(BATCH, NH, sq, sk, device=DEV,
                                      dtype=torch.bfloat16),
             "scale": SCALE})

# Prefill: full self-attention, is_causal, no explicit mask.
PREFILL_CASES = [(f"seq_{s}", s, s) for s in (128, 512)]

def prefill_inputs(sq, sk):
    return ((rand(BATCH, NH, sq, HEAD_DIM), rand(BATCH, NKV, sk, HEAD_DIM),
             rand(BATCH, NKV, sk, HEAD_DIM)),
            {"is_causal": True, "scale": SCALE})

## mm

`C = A @ B`. The unverified GEMMs are addmm-shaped, so they show up in the next section.

In [3]:
bench_matrix(MM_CASES, mm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("tc2d_to", kuipy.run(aten.mm.default, impl="tc2d_to")),
              ("gemm_tc", gemm_tc_mm)],
             torch.mm, flops=GEMM_FLOPS)

,case,M,K,N,tc2d_to GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,tc2d_to us,gemm_tc us,ref us,tc2d_to rel-err,gemm_tc rel-err
0,o_proj,256,896,896,3953.979510,17904.013674,33457.637095,103.956480,22.958081,12.285440,0.000000,0.000000
1,gate_proj,256,896,4864,31111.822790,60162.118620,59963.456837,71.720958,37.089281,37.212160,0.002713,0.002713
2,up_proj,256,896,4864,31112.099967,59151.625706,59570.039671,71.720319,37.722881,37.457919,0.002711,0.002711
3,down_proj,256,4864,896,6059.668729,42560.001075,65674.263853,368.232956,52.428799,33.976319,0.002622,0.002623
4,lm_head,256,896,151936,43016.335920,73568.800904,100278.923304,1620.336609,947.425308,695.070724,0.000000,0.000000
5,square_4096,4096,4096,4096,43871.753989,74907.482156,116348.866234,3132.743530,1834.782715,1181.265945,0.000000,0.000000


In [4]:
bench_matrix(MM_CASES, mm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("tc2d_to", kuipy.run(aten.mm.default, impl="tc2d_to")),
              ("tc2d", kuipy.run(aten.mm.default, impl="tc2d",
                                 acc_dtype=torch.float16)),
              ("gemm_tc", gemm_tc_mm)],
             torch.mm, flops=GEMM_FLOPS)

,case,M,K,N,tc2d_to GFLOP/s,tc2d GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,tc2d_to us,tc2d us,gemm_tc us,ref us,tc2d_to rel-err,tc2d rel-err,gemm_tc rel-err
0,o_proj,256,896,896,1943.676170,2013.685116,19638.956039,16082.857464,211.476479,204.124165,20.929921,25.557759,0.000000,0.001573,0.000000
1,gate_proj,256,896,4864,20924.448523,28088.064172,59898.553575,51612.316221,106.639357,79.441919,37.252481,43.233280,0.000340,0.001588,0.000340
2,up_proj,256,896,4864,28146.112798,28051.905215,65634.700741,56481.907138,79.278078,79.544320,33.996799,39.505920,0.000339,0.001584,0.000339
3,down_proj,256,4864,896,6322.748071,6748.024080,48858.116949,66027.481922,352.911377,330.670090,45.670400,33.794560,0.000329,0.003622,0.000329
4,lm_head,256,896,151936,45421.823272,45451.548316,85844.016164,100036.946368,1534.525452,1533.521881,811.948776,696.752014,0.000000,0.001575,0.000000
5,square_4096,4096,4096,4096,47041.782618,48505.184047,89564.617943,111928.319428,2921.635742,2833.489990,1534.522858,1227.919388,0.000000,0.003331,0.000000


## addmm

`D = beta*C + alpha*(A @ B)`: the full epilogue, which the bias-free `mm` path never hits.
`gemm_pipe` is fp16-only and `gemm_hacky_epilogue` bf16-only, hence the two tables.
`gemm_tc` handles both, and takes its epilogue term as an (M, N) matrix or a length-N
vector, so it is the only contender that appears in all three.

In [5]:
bench_matrix(GEMM_CASES, addmm_inputs(torch.bfloat16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("hacky_epilogue", unverified.gemm_hacky_epilogue),
              ("gemm_tc", unverified.gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,hacky_epilogue GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,hacky_epilogue us,gemm_tc us,ref us,kuiper rel-err,hacky_epilogue rel-err,gemm_tc rel-err
0,o_proj,256,896,896,3779.024551,4186.566382,21958.862752,11243.921750,108.769283,98.181124,18.718719,36.556799,0.000004,0.000004,0.000004
1,down_proj,256,4864,896,5626.896912,5926.222294,42197.364702,57739.055190,396.554222,376.524811,52.879362,38.645761,0.002559,0.002559,0.002560
2,square_4096,4096,4096,4096,42853.407928,42538.883017,71676.875148,105098.994187,3207.188416,3230.901794,1917.479706,1307.709503,0.000005,0.000005,0.000005


In [6]:
bench_matrix(GEMM_CASES, addmm_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", kuipy.run(aten.addmm.default)),
              ("gemm_pipe", unverified.gemm_pipe),
              ("gemm_tc", unverified.gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,gemm_pipe GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,kuiper us,gemm_pipe us,gemm_tc us,ref us,kuiper rel-err,gemm_pipe rel-err,gemm_tc rel-err
0,o_proj,256,896,896,4120.412290,6981.008682,24357.281911,11461.841285,99.757442,58.880000,16.875520,35.861759,8.420163e-07,8.420163e-07,8.420163e-07
1,down_proj,256,4864,896,5753.477580,12930.835636,47186.489972,57861.709453,387.829742,172.561913,47.288318,38.563840,3.200818e-04,3.200818e-04,3.198812e-04
2,square_4096,4096,4096,4096,46256.136004,68846.552285,85023.268361,102440.645333,2971.258850,1996.308441,1616.486359,1341.644745,1.462452e-06,1.462452e-06,1.462452e-06


### broadcast bias

`nn.Linear` emits `addmm(bias, x, W.T)` with `bias` a length-N *vector*, not an (M, N)
matrix. A tlayout is an injection, so the stride-0 row axis of a broadcast C is
inexpressible as one; C is now read as an `rotensor` over a *virtual* tensor layout,
which need not be injective, and `TensorCore2D.To`'s out-of-place epilogue reads C and
writes D through independent index functions, so dropping C's row term costs one term in
the index expression. `kuiper` is that verified path. `gemm_bcast_bias_epilogue` and
`gemm_bcast_bias_epilogue2` are the hand-edited extractions that prototyped it (the
former stages the bias slice into shared memory replicated over a fragment's rows, the
latter is the one-line lift of `TensorCore2D.To`). `kuiper+materialise` is what the
verified path had to do before: materialise the full (M, N) C and run the stock kernel.

In [7]:
def bias_inputs(dtype):
    return lambda M, K, N: ((rand(N, dtype=dtype), rand(M, K, dtype=dtype),
                             rand(K, N, dtype=dtype)), {"beta": BETA, "alpha": ALPHA})

_kuiper_addmm = kuipy.run(aten.addmm.default)

def kuiper_dense_bias(bias, A, B, beta=1.0, alpha=1.0):
    return _kuiper_addmm(bias.expand(A.size(0), B.size(1)).contiguous(), A, B,
                         beta=beta, alpha=alpha)

bench_matrix(GEMM_CASES, bias_inputs(torch.float16), MNK, ["M", "K", "N"],
             [("kuiper", _kuiper_addmm),
              ("kuiper+materialise", kuiper_dense_bias),
              ("bcast_epilogue", unverified.gemm_bcast_bias_epilogue),
              ("bcast_epilogue2", unverified.gemm_bcast_bias_epilogue2),
              ("gemm_tc", unverified.gemm_tc)],
             torch.addmm, flops=GEMM_FLOPS)

,case,M,K,N,kuiper GFLOP/s,kuiper+materialise GFLOP/s,bcast_epilogue GFLOP/s,bcast_epilogue2 GFLOP/s,gemm_tc GFLOP/s,ref GFLOP/s,...,kuiper+materialise us,bcast_epilogue us,bcast_epilogue2 us,gemm_tc us,ref us,kuiper rel-err,kuiper+materialise rel-err,bcast_epilogue rel-err,bcast_epilogue2 rel-err,gemm_tc rel-err
0,o_proj,256,896,896,4198.828385,4714.681644,5639.336747,5719.692312,25534.860389,11799.176746,...,87.183361,72.888322,71.864319,16.097280,34.836481,0.000004,0.000004,0.000000,0.000004,0.000004
1,down_proj,256,4864,896,6345.210243,6374.165209,6482.246757,6500.811806,48770.635322,58046.671044,...,350.064621,344.227829,343.244781,45.752320,38.440959,0.000318,0.000318,0.000318,0.000318,0.000318
2,square_4096,4096,4096,4096,46283.912376,46319.671868,46957.893640,47622.009493,89758.528837,104896.940829,...,2967.183228,2926.855164,2886.038513,1531.207733,1310.228424,0.000001,0.000001,0.000000,0.000001,0.000001


## sdpa

The decode mask is a dense `(B, Hq, Sq, Sk)` tensor: the Kuiper mask is read through a
`rotensor`, whose layout need not be an injection, but the instantiation template only
emits the dense row-major layout, so the mask is materialised and handed to every
contender for fairness. Prefill passes no mask at all, which the kernel selects with a
broadcast layout plus `has_mask = false`.

In [8]:
bench_matrix(DECODE_CASES, decode_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("kuiper", kuiper_sdpa),
              ("manual_extract", unverified.flash_attn_manual_extract),
              ("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

,case,Sq,Sk,kuiper GFLOP/s,manual_extract GFLOP/s,fa1 GFLOP/s,fa2 GFLOP/s,ref GFLOP/s,kuiper us,manual_extract us,fa1 us,fa2 us,ref us,kuiper rel-err,manual_extract rel-err,fa1 rel-err,fa2 rel-err
0,ctx_128,1,128,622.627591,791.825441,896.840817,2455.845735,2403.352884,188.620796,148.316164,130.949116,47.820802,48.865280,0.002320,0.002320,0.002320,0.001661
1,ctx_512,1,512,1246.744216,1252.120779,1216.915387,3418.929747,3837.002416,376.791039,375.173111,386.026878,137.400322,122.429438,0.002361,0.002361,0.002361,0.001629
2,ctx_1024,1,1024,1346.498384,1350.025048,1268.184856,3643.780661,3964.327811,697.753601,695.930862,740.841599,257.843208,236.994553,0.002336,0.002336,0.002336,0.001464
3,ctx_16384,1,16384,1539.849644,1543.534663,1395.950267,3965.808445,4043.919406,9762.242432,9738.936157,10768.568115,3790.497131,3717.281189,0.002330,0.002330,0.002330,0.000875


In [9]:
bench_matrix(PREFILL_CASES, prefill_inputs, lambda sq, sk: (sq, sk), ["Sq", "Sk"],
             [("kuiper", kuiper_sdpa),
              ("fa1", unverified.flash_attn_fa1),
              ("fa2", unverified.flash_attn_fa2)],
             cudnn_sdpa, flops=attn_flops)

,case,Sq,Sk,kuiper GFLOP/s,fa1 GFLOP/s,fa2 GFLOP/s,ref GFLOP/s,kuiper us,fa1 us,fa2 us,ref us,kuiper rel-err,fa1 rel-err,fa2 rel-err
0,seq_128,128,128,3283.454884,5374.202204,16201.730396,65157.849854,4578.222046,2797.138062,927.825928,230.707207,0.001301,0.000928,0.000928
1,seq_512,512,512,7415.457351,9571.346447,35766.760331,117160.496989,32434.704590,25128.979492,6724.628296,2052.894745,0.001562,0.001075,0.001075
